# SemEval-2026 Task 13 — Subtask A: GraphCodeBERT

**Binary Classification** — Detect machine-generated vs human-written code.

### Model: `microsoft/graphcodebert-base`
- Pre-trained with data flow information → structure-aware representations
- Same RoBERTa architecture as CodeBERT → drop-in replacement
- Hypothesis: DFG-aware pretraining improves code understanding

### Setup
- **GPU:** Kaggle Tesla T4 (16 GB VRAM)
- **Data:** 10% stratified subsample of training set
- **Metric:** Macro F1 (competition metric)

Set runtime to `GPU T4 x2` or `GPU T4` in Kaggle settings.

In [1]:
# ============================================================
# Cell 1: Environment Setup
# ============================================================
!pip install torch==2.1.2 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install --upgrade transformers==4.45.0 huggingface_hub datasets scikit-learn accelerate -q


Looking in indexes: https://download.pytorch.org/whl/cu118
ERROR: Could not find a version that satisfies the requirement torch==2.1.2 (from versions: 2.2.0+cu118, 2.2.1+cu118, 2.2.2+cu118, 2.3.0+cu118, 2.3.1+cu118, 2.4.0+cu118, 2.4.1+cu118, 2.5.0+cu118, 2.5.1+cu118, 2.6.0+cu118, 2.7.0+cu118, 2.7.1+cu118)
ERROR: No matching distribution found for torch==2.1.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 133.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 143.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 101.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 125.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 123.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 74.1 kB/s eta 0:00:00


In [2]:
# ============================================================
# Cell 2: Imports
# ============================================================
import os
os.environ["WANDB_DISABLED"] = "true"

import gc
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from datasets import Dataset, load_dataset
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
)
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    f1_score,
)
import warnings
warnings.filterwarnings("ignore")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', getattr(props, 'total_mem', 0))
    print(f"VRAM: {vram / 1e9:.1f} GB")

2026-04-05 10:31:54.639371: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775385114.825249      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775385114.877537      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775385115.309790      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775385115.309828      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775385115.309831      24 computation_placer.cc:177] computation placer alr

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


## Configuration

In [3]:
# ============================================================
# Cell 3: Configuration
# ============================================================

# --- Model ---
MODEL_NAME = "microsoft/graphcodebert-base"

from datasets import load_dataset
print("Downloading SemEval-2026-Task13 Subtask B from HuggingFace...")
hf_dataset = load_dataset("DaniilOr/SemEval-2026-Task13", "A")
hf_dataset['train'].to_parquet('task_a_training_set_1.parquet')
hf_dataset['validation'].to_parquet('task_a_validation_set.parquet')
if 'test' in hf_dataset:
    hf_dataset['test'].to_parquet('task_a_test_set_sample.parquet')
    TEST_PATH = 'task_a_test_set_sample.parquet'
else:
    TEST_PATH = None
    print("No test split available on HuggingFace.")
TRAIN_PATH = 'task_a_training_set_1.parquet'
VAL_PATH   = 'task_a_validation_set.parquet'
print("Done!")

print(f"Train: {TRAIN_PATH}")
print(f"Val:   {VAL_PATH}")
print(f"Test:  {TEST_PATH}")

# --- Subsampling (10% of ~500K dataset) ---
TRAIN_SAMPLE_SIZE = 50000       # 10% of 500K
VAL_SAMPLE_SIZE   = 12500       # 25% of train sample
RANDOM_SEED       = 42

# --- Training hyperparameters ---
MAX_LENGTH         = 256         # Sequence length (T4-safe)
BATCH_SIZE         = 16
GRAD_ACCUM_STEPS   = 2           # Effective batch = 32
LEARNING_RATE      = 2e-5
NUM_EPOCHS         = 10
WEIGHT_DECAY       = 0.01
WARMUP_RATIO       = 0.1
LABEL_SMOOTHING    = 0.0         # Task A is balanced → no smoothing needed
EARLY_STOPPING_PATIENCE = 5

# --- Output ---
OUTPUT_DIR = "/kaggle/working/results_graphcodebert_taskA"

print(f"Model: {MODEL_NAME}")
print(f"Max length: {MAX_LENGTH}")
print(f"Train sample: {TRAIN_SAMPLE_SIZE} (10% of dataset), Val sample: {VAL_SAMPLE_SIZE}")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"Learning rate: {LEARNING_RATE}")

README.md:   0%|          | 0.00/801 [00:00<?, ?B/s]

task_a/task_a_training_set_1.parquet:   0%|          | 0.00/203M [00:00<?, ?B/s]

task_a/task_a_validation_set.parquet:   0%|          | 0.00/40.5M [00:00<?, ?B/s]

task_a/task_a_test_set_sample.parquet:   0%|          | 0.00/593k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/500000 [00:00<?, ? examples/s]

...

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Done!
Train: task_a_training_set_1.parquet
Val:   task_a_validation_set.parquet
Test:  task_a_test_set_sample.parquet
Model: microsoft/graphcodebert-base
Max length: 256
Train sample: 50000 (10% of dataset), Val sample: 12500
Effective batch size: 32
Learning rate: 2e-05


## GraphCodeBERT Trainer Class

Follows the baseline `CodeClassifierTrainer` structure with:
- GraphCodeBERT backbone (`microsoft/graphcodebert-base`)
- Macro F1 as the primary metric (competition metric)
- Cosine LR scheduler + early stopping
- fp16 mixed precision for T4

In [4]:
# ============================================================
# Cell 4: GraphCodeBERT Trainer (follows baseline structure)
# ============================================================

class GraphCodeBERTTrainerA:
    """
    Task A trainer using GraphCodeBERT.
    Mirrors the baseline CodeClassifierTrainer pipeline:
      load_and_prepare_data → initialize_model_and_tokenizer →
      prepare_datasets → train → evaluate_model
    """

    def __init__(self, model_name=MODEL_NAME, max_length=MAX_LENGTH):
        self.model_name = model_name
        self.max_length = max_length
        self.tokenizer = None
        self.model = None
        self.num_labels = None
        self.tag = model_name.split("/")[-1]

    # ------------------------------------------------------------------
    # Data loading + stratified subsampling (10%)
    # ------------------------------------------------------------------
    def load_and_prepare_data(
        self,
        sample_size=TRAIN_SAMPLE_SIZE,
        val_sample_size=VAL_SAMPLE_SIZE,
        random_seed=RANDOM_SEED,
    ):
        try:
            df = pd.read_parquet(TRAIN_PATH)
            print(f"[{self.tag}] Dataset columns: {df.columns.tolist()}")
            print(f"[{self.tag}] Original dataset size: {len(df)}")

            if 'code' not in df.columns or 'label' not in df.columns:
                raise ValueError("Dataset must contain 'code' and 'label' columns")

            df = df.dropna(subset=['code', 'label'])
            df['label'] = df['label'].astype(int)

            # --- Stratified subsampling (10%) ---
            if sample_size is not None and sample_size < len(df):
                df = df.groupby('label', group_keys=False).apply(
                    lambda x: x.sample(
                        n=max(1, int(sample_size * len(x) / len(df))),
                        random_state=random_seed,
                    )
                ).reset_index(drop=True)
                print(f"[{self.tag}] Sampled train size: {len(df)} ({len(df)*100/500000:.1f}% of full dataset, stratified, seed={random_seed})")

            self.num_labels = df['label'].nunique()
            print(f"[{self.tag}] Number of unique labels: {self.num_labels}")
            print(f"[{self.tag}] Train label distribution:")
            print(df['label'].value_counts().sort_index())

            # --- Validation set ---
            val_df = pd.read_parquet(VAL_PATH)
            val_df = val_df.dropna(subset=['code', 'label'])
            val_df['label'] = val_df['label'].astype(int)

            if val_sample_size is not None and val_sample_size < len(val_df):
                val_df = val_df.groupby('label', group_keys=False).apply(
                    lambda x: x.sample(
                        n=max(1, int(val_sample_size * len(x) / len(val_df))),
                        random_state=random_seed,
                    )
                ).reset_index(drop=True)
                print(f"[{self.tag}] Sampled val size: {len(val_df)} (stratified, seed={random_seed})")

            print(f"[{self.tag}] Val label distribution:")
            print(val_df['label'].value_counts().sort_index())
            print(f"[{self.tag}] Train: {len(df)},  Val: {len(val_df)}")
            return df, val_df

        except Exception as e:
            print(f"[{self.tag}] Error loading dataset: {e}")
            raise

    # ------------------------------------------------------------------
    # Model + tokenizer initialization
    # ------------------------------------------------------------------
    def initialize_model_and_tokenizer(self):
        print(f"[{self.tag}] Initialising model and tokenizer ...")

        # GraphCodeBERT uses the same RobertaTokenizer as CodeBERT
        self.tokenizer = RobertaTokenizer.from_pretrained(self.model_name)

        # Standard sequence classification head on top of GraphCodeBERT
        self.model = RobertaForSequenceClassification.from_pretrained(
            self.model_name,
            num_labels=self.num_labels,
            problem_type="single_label_classification",
        ).to('cuda')

        total = sum(p.numel() for p in self.model.parameters())
        train_p = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        print(f"[{self.tag}] {self.num_labels} labels | params {total:,} | trainable {train_p:,}")

    # ------------------------------------------------------------------
    # Tokenization
    # ------------------------------------------------------------------
    def tokenize_function(self, examples):
        return self.tokenizer(
            examples['code'],
            truncation=True,
            padding=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

    # ------------------------------------------------------------------
    # Dataset preparation
    # ------------------------------------------------------------------
    def prepare_datasets(self, train_df, val_df):
        print(f"[{self.tag}] Preparing datasets ...")
        train_dataset = Dataset.from_pandas(train_df[['code', 'label']])
        val_dataset   = Dataset.from_pandas(val_df[['code', 'label']])

        train_dataset = train_dataset.map(
            self.tokenize_function, batched=True, remove_columns=['code']
        )
        val_dataset = val_dataset.map(
            self.tokenize_function, batched=True, remove_columns=['code']
        )

        train_dataset = train_dataset.rename_column('label', 'labels')
        val_dataset   = val_dataset.rename_column('label', 'labels')

        print(f"[{self.tag}] Train: {len(train_dataset)}, Val: {len(val_dataset)}")
        return train_dataset, val_dataset

    # ------------------------------------------------------------------
    # Metrics — Macro F1 (competition metric)
    # ------------------------------------------------------------------
    def compute_metrics(self, eval_pred):
        predictions, labels = eval_pred
        preds = np.argmax(predictions, axis=1)

        accuracy = accuracy_score(labels, preds)
        macro_f1 = f1_score(labels, preds, average='macro', zero_division=0)
        precision, recall, f1_w, _ = precision_recall_fscore_support(
            labels, preds, average='weighted', zero_division=0
        )

        return {
            'accuracy': accuracy,
            'macro_f1': macro_f1,
            'f1': f1_w,
            'precision': precision,
            'recall': recall,
        }

    # ------------------------------------------------------------------
    # Training
    # ------------------------------------------------------------------
    def train(
        self,
        train_dataset,
        val_dataset,
        output_dir=OUTPUT_DIR,
        num_epochs=NUM_EPOCHS,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
    ):
        print(f"[{self.tag}] Starting training ...")

        steps_per_epoch = len(train_dataset) // (batch_size * GRAD_ACCUM_STEPS)
        eval_save_steps = max(200, steps_per_epoch // 2)

        training_args = TrainingArguments(
            output_dir=output_dir,
            num_train_epochs=num_epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size * 2,
            gradient_accumulation_steps=GRAD_ACCUM_STEPS,
            weight_decay=WEIGHT_DECAY,
            warmup_ratio=WARMUP_RATIO,
            logging_dir=f"{output_dir}/logs",
            logging_steps=50,
            eval_strategy="steps",
            eval_steps=eval_save_steps,
            save_strategy="steps",
            save_steps=eval_save_steps,
            load_best_model_at_end=True,
            metric_for_best_model="macro_f1",
            greater_is_better=True,
            remove_unused_columns=False,
            learning_rate=learning_rate,
            lr_scheduler_type="cosine",
            save_total_limit=2,
            report_to=[],
            fp16=True,
        )

        data_collator = DataCollatorWithPadding(tokenizer=self.tokenizer)

        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            tokenizer=self.tokenizer,
            data_collator=data_collator,
            compute_metrics=self.compute_metrics,
            callbacks=[EarlyStoppingCallback(
                early_stopping_patience=EARLY_STOPPING_PATIENCE
            )],
        )

        print(f"[{self.tag}] Config:")
        print(f"  Train: {len(train_dataset)}, Val: {len(val_dataset)}")
        print(f"  Epochs: {num_epochs}, Batch: {batch_size}x{GRAD_ACCUM_STEPS}")
        print(f"  LR: {learning_rate}, MaxLen: {self.max_length}")
        print(f"  Scheduler: cosine, fp16: True")
        print(f"  Eval every {eval_save_steps} steps")

        trainer.train()

        trainer.save_model()
        self.tokenizer.save_pretrained(output_dir)
        print(f"[{self.tag}] Training completed. Model saved to {output_dir}")
        return trainer

    # ------------------------------------------------------------------
    # Evaluation
    # ------------------------------------------------------------------
    def evaluate_model(self, trainer, val_dataset):
        print(f"\n{'='*60}")
        print(f"  EVALUATION — {self.tag}")
        print(f"{'='*60}")

        predictions = trainer.predict(val_dataset)
        y_pred = np.argmax(predictions.predictions, axis=1)
        y_true = predictions.label_ids

        # Per-class report
        print("\nPer-Class Classification Report:")
        print(classification_report(
            y_true, y_pred,
            target_names=["human", "machine"],
            zero_division=0,
        ))

        # Competition metric
        macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
        print(f"\n>>> COMPETITION METRIC (Macro F1): {macro_f1:.4f} <<<")
        return predictions

    # ------------------------------------------------------------------
    # Full pipeline
    # ------------------------------------------------------------------
    def run_full_pipeline(
        self,
        output_dir=OUTPUT_DIR,
        num_epochs=NUM_EPOCHS,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        sample_size=TRAIN_SAMPLE_SIZE,
        val_sample_size=VAL_SAMPLE_SIZE,
        random_seed=RANDOM_SEED,
    ):
        try:
            train_df, val_df = self.load_and_prepare_data(
                sample_size=sample_size,
                val_sample_size=val_sample_size,
                random_seed=random_seed,
            )
            self.initialize_model_and_tokenizer()
            train_dataset, val_dataset = self.prepare_datasets(train_df, val_df)
            trainer = self.train(
                train_dataset, val_dataset,
                output_dir=output_dir,
                num_epochs=num_epochs,
                batch_size=batch_size,
                learning_rate=learning_rate,
            )
            self.evaluate_model(trainer, val_dataset)
            print(f"\n[{self.tag}] Pipeline completed successfully!")
            return trainer
        except Exception as e:
            print(f"[{self.tag}] Error in pipeline: {e}")
            raise

print("GraphCodeBERTTrainerA defined.")

GraphCodeBERTTrainerA defined.


## Test Set Evaluation

Runs inference on the test parquet and prints per-category metrics
(seen/unseen language × seen/unseen domain).

In [5]:
# ============================================================
# Cell 5: Test-set evaluation utilities (from baseline)
# ============================================================
from tqdm import tqdm

SEEN_LANGS   = {'c++', 'cpp', 'python', 'java'}
UNSEEN_LANGS = {'go', 'php', 'c#', 'csharp', 'c', 'javascript', 'js'}
SEEN_DOMAINS   = {'algorithmic'}
UNSEEN_DOMAINS = {'research', 'production'}


@torch.no_grad()
def evaluate_on_test(trainer_obj, parquet_path, max_length=256,
                     batch_size=32, device=None):
    """Run inference on test parquet → return DataFrame with 'prediction' column."""
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = trainer_obj.tokenizer
    model = trainer_obj.model
    model.to(device)
    model.eval()

    df = pd.read_parquet(parquet_path)
    df = df.dropna(subset=['code']).reset_index(drop=True)
    codes = df['code'].tolist()

    preds = []
    for i in tqdm(range(0, len(codes), batch_size), desc="Predicting"):
        batch = codes[i:i + batch_size]
        enc = tokenizer(
            batch, truncation=True, padding=True,
            max_length=max_length, return_tensors="pt",
        )
        logits = model(
            input_ids=enc["input_ids"].to(device),
            attention_mask=enc["attention_mask"].to(device),
        ).logits
        preds.extend(logits.argmax(dim=-1).cpu().tolist())

    df['prediction'] = preds
    return df


def evaluate_by_category(df, tag="GraphCodeBERT"):
    """Print classification metrics for 4 evaluation settings + overall."""
    lang_col = next(
        (c for c in df.columns if c.lower() in ('language', 'lang', 'programming_language')),
        None,
    )
    domain_col = next(
        (c for c in df.columns if c.lower() in ('domain', 'task_type', 'category')),
        None,
    )

    if 'label' not in df.columns:
        print(f"[{tag}] No 'label' column — cannot evaluate.")
        return

    # Overall metrics (always print)
    y_true, y_pred = df['label'].values, df['prediction'].values
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    acc = accuracy_score(y_true, y_pred)

    print(f"\n{'='*70}")
    print(f"  TEST RESULTS  --  {tag}")
    print(f"{'='*70}")

    print("\nOverall Classification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))
    print(f"  OVERALL  (n={len(df)})  Accuracy={acc:.4f}  Macro F1={macro_f1:.4f}")

    # Per-category breakdown (if columns exist)
    if lang_col is not None and domain_col is not None:
        _norm = lambda v: str(v).strip().lower()
        df['_l'] = df[lang_col].apply(_norm)
        df['_d'] = df[domain_col].apply(_norm)

        settings = [
            ("(i)   Seen Lang & Seen Domain",      SEEN_LANGS,   SEEN_DOMAINS),
            ("(ii)  Unseen Lang & Seen Domain",    UNSEEN_LANGS, SEEN_DOMAINS),
            ("(iii) Seen Lang & Unseen Domain",    SEEN_LANGS,   UNSEEN_DOMAINS),
            ("(iv)  Unseen Lang & Unseen Domain",  UNSEEN_LANGS, UNSEEN_DOMAINS),
        ]

        for name, langs, domains in settings:
            mask = df['_l'].isin(langs) & df['_d'].isin(domains)
            sub = df[mask]
            n = len(sub)
            if n == 0:
                print(f"\n  {name}:  ** no samples **")
                continue
            y_t, y_p = sub['label'].values, sub['prediction'].values
            sub_acc = accuracy_score(y_t, y_p)
            sub_mf1 = f1_score(y_t, y_p, average='macro', zero_division=0)
            p, r, f1, _ = precision_recall_fscore_support(
                y_t, y_p, average='weighted', zero_division=0
            )
            print(f"\n  {name}  (n={n})")
            print(f"    Accuracy={sub_acc:.4f}  Macro F1={sub_mf1:.4f}")
            print(classification_report(y_t, y_p, zero_division=0))

        df.drop(columns=['_l', '_d'], inplace=True)

    print("=" * 70)

print("evaluate_on_test(), evaluate_by_category() defined.")

evaluate_on_test(), evaluate_by_category() defined.


---
## Run Training + Evaluation

In [6]:
# ============================================================
# Cell 6: Run GraphCodeBERT on Task A
# ============================================================

print("\n" + "=" * 70)
print("  GraphCodeBERT — Task A (Binary Classification)")
print("=" * 70)

trainer_obj = GraphCodeBERTTrainerA(
    model_name=MODEL_NAME,
    max_length=MAX_LENGTH,
)

hf_trainer = trainer_obj.run_full_pipeline(
    output_dir=OUTPUT_DIR,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    sample_size=TRAIN_SAMPLE_SIZE,
    val_sample_size=VAL_SAMPLE_SIZE,
    random_seed=RANDOM_SEED,
)


  GraphCodeBERT — Task A (Binary Classification)
[graphcodebert-base] Dataset columns: ['code', 'generator', 'label', 'language']
[graphcodebert-base] Original dataset size: 500000
[graphcodebert-base] Sampled train size: 49999 (10.0% of full dataset, stratified, seed=42)
[graphcodebert-base] Number of unique labels: 2
[graphcodebert-base] Train label distribution:
label
0    23847
1    26152
Name: count, dtype: int64
[graphcodebert-base] Sampled val size: 12499 (stratified, seed=42)
[graphcodebert-base] Val label distribution:
label
0    5961
1    6538
Name: count, dtype: int64
[graphcodebert-base] Train: 49999,  Val: 12499
[graphcodebert-base] Initialising model and tokenizer ...


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

...

Map:   0%|          | 0/12499 [00:00<?, ? examples/s]

[graphcodebert-base] Train: 49999, Val: 12499
[graphcodebert-base] Starting training ...
[graphcodebert-base] Config:
  Train: 49999, Val: 12499
  Epochs: 10, Batch: 16x2
  LR: 2e-05, MaxLen: 256
  Scheduler: cosine, fp16: True
  Eval every 781 steps


Step,Training Loss,Validation Loss,Accuracy,Macro F1,F1,Precision,Recall
781,0.077900,0.063055,0.983999,0.983958,0.983995,0.984031,0.983999
1562,0.131100,0.072274,0.986879,0.986852,0.986879,0.986880,0.986879
2343,0.081600,0.148347,0.980478,0.980461,0.980488,0.981020,0.980478
3124,0.148500,0.154872,0.987519,0.987496,0.987521,0.987543,0.987519
3905,0.056300,0.149479,0.988479,0.988452,0.988478,0.988487,0.988479
4686,0.027600,0.157038,0.989039,0.989017,0.989040,0.989046,0.989039
5467,0.018100,0.170577,0.988479,0.988452,0.988478,0.988485,0.988479
6248,0.012200,0.165083,0.989199,0.989176,0.989199,0.989199,0.989199
7029,0.022200,0.167396,0.989519,0.989498,0.989520,0.989525,0.989519
7810,0.008300,0.169104,0.989279,0.989258,0.989280,0.989284,0.989279


[graphcodebert-base] Training completed. Model saved to /kaggle/working/results_graphcodebert_taskA

  EVALUATION — graphcodebert-base



Per-Class Classification Report:
              precision    recall  f1-score   support

       human       0.99      0.99      0.99      5961
     machine       0.99      0.99      0.99      6538

    accuracy                           0.99     12499
   macro avg       0.99      0.99      0.99     12499
weighted avg       0.99      0.99      0.99     12499


>>> COMPETITION METRIC (Macro F1): 0.9895 <<<

[graphcodebert-base] Pipeline completed successfully!


In [7]:
# ============================================================
# Cell 7: Evaluate on test set
# ============================================================

import os
if os.path.exists(TEST_PATH):
    test_df = evaluate_on_test(
        trainer_obj=trainer_obj,
        parquet_path=TEST_PATH,
        max_length=MAX_LENGTH,
        batch_size=BATCH_SIZE * 2,
        device="cuda",
    )
    evaluate_by_category(test_df, tag="GraphCodeBERT-TaskA")
else:
    print(f"Test file not found at {TEST_PATH}. Skipping test evaluation.")

Predicting: 100%|██████████| 32/32 [00:16<00:00,  1.90it/s]


  TEST RESULTS  --  GraphCodeBERT-TaskA

Overall Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.19      0.31       777
           1       0.23      0.84      0.36       223

    accuracy                           0.34      1000
   macro avg       0.52      0.52      0.34      1000
weighted avg       0.68      0.34      0.32      1000

  OVERALL  (n=1000)  Accuracy=0.3370  Macro F1=0.3360


In [8]:
# ============================================================
# Cell 8: Clean up GPU memory
# ============================================================

del hf_trainer, trainer_obj
gc.collect()
torch.cuda.empty_cache()
print("GPU memory freed.")

GPU memory freed.
